<a href="https://colab.research.google.com/github/siddhartha-sai-17/Gen-AI/blob/main/Intelligent_Employee_Policy_Assistant_%E2%80%93_Chunking_Analysis_Problem_Statement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Intelligent Employee Policy Assistant – Chunking Analysis

In [1]:
# Install necessary libraries
!pip install reportlab PyPDF2 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 11.5 MB/s eta 0:00:00


### Task 1: PDF Loading

**Objective:** Load and extract text from the PDF.

In [2]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from PyPDF2 import PdfReader
import os

def create_policy_pdf(filename="policy_document.pdf"):
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter

    text_content = [
        "Leave Policy",
        "Employees receive 12 casual leaves annually.",
        "Employees receive 15 sick leaves annually.",
        "Unused casual leaves cannot be carried forward.",
        "",
        "Work From Home Policy",
        "Employees may work from home twice per week.",
        "Manager approval is required for additional remote work.",
        "",
        "Travel Policy",
        "Travel expenses are reimbursed within 30 days.",
        "Original receipts must be submitted for reimbursement.",
        "",
        "Medical Insurance Policy",
        "All employees are covered under company medical insurance.",
        "Dependent coverage is available for spouse and children."
    ]

    y_position = height - 50
    for line in text_content:
        c.drawString(50, y_position, line)
        y_position -= 15 # Move to the next line

    c.save()
    print(f"PDF '{filename}' created successfully.")

# Create the PDF document
create_policy_pdf()


PDF 'policy_document.pdf' created successfully.


In [3]:
def extract_text_from_pdf(pdf_path="policy_document.pdf"):
    text = ""
    if not os.path.exists(pdf_path):
        print(f"Error: PDF file not found at {pdf_path}")
        return ""
    try:
        with open(pdf_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                text += page.extract_text() + "\n"
    except Exception as e:
        print(f"Error reading PDF: {e}")
    return text

# Extract text
original_document_text = extract_text_from_pdf()

# Display Deliverables
print("Original Document:\n")
print(original_document_text)
print(f"\nTotal Characters: {len(original_document_text)}")
print(f"Total Words: {len(original_document_text.split())}")


Original Document:

Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.



Total Characters: 531
Total Words: 76


### Task 2: Fixed Size Chunking

**Objective:** Implement Fixed Size Chunking.

In [4]:
def fixed_size_chunking(text, chunk_size):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])
    return chunks

chunk_sizes = [100, 200, 300]

for size in chunk_sizes:
    print(f"\n--- Fixed Size Chunking (Chunk Size = {size}) ---")
    chunks = fixed_size_chunking(original_document_text, size)
    for i, chunk in enumerate(chunks):
        print(f"Chunk Number: {i+1}")
        print(f"Chunk Content: {chunk.strip()}")
        print(f"Chunk Length: {len(chunk)}")
        print("\n")



--- Fixed Size Chunking (Chunk Size = 100) ---
Chunk Number: 1
Chunk Content: Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Chunk Length: 100


Chunk Number: 2
Chunk Content: Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home
Chunk Length: 100


Chunk Number: 3
Chunk Content: twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expens
Chunk Length: 100


Chunk Number: 4
Chunk Content: es are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Ins
Chunk Length: 100


Chunk Number: 5
Chunk Content: urance Policy
All employees are covered under company medical insurance.
Dependent coverage is avail
Chunk Length: 100


Chunk Number: 6
Chunk Content: able for spouse and children.
Chunk Length: 31



--- Fixed Size Chunking (Chunk Size = 200) ---
Chunk Number: 1
Chunk Content: Leave Policy
Employees receive 12 c

#### Analysis of Fixed Size Chunking

**Are sentences being split?**
Yes, fixed-size chunking often splits sentences in the middle if the chunk boundary falls within a sentence, especially with smaller chunk sizes.

**Is context preserved?**
Context preservation can be poor. When sentences or logical units are split, the chunk might lose its meaning or be incomplete, making it harder for a retrieval system to understand the full context.

**What are the limitations?**
1.  **Arbitrary Splits:** Chunks can end mid-sentence or mid-word, breaking linguistic integrity.
2.  **Loss of Context:** Important contextual information can be separated across chunks, leading to less effective retrieval.
3.  **Suboptimal for Meaningful Units:** It doesn't consider the semantic boundaries of the text, such as paragraphs, sections, or complete thoughts.

### Task 3: Recursive Chunking

**Objective:** Implement Recursive Character Chunking using LangChain.

In [5]:
# Install LangChain and tiktoken
!pip install -q langchain langchain_text_splitters tiktoken


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_chunking(text, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )
    chunks = text_splitter.create_documents([text])
    return chunks

# Experiment with different chunk sizes and overlaps
recursive_configs = [
    {"chunk_size": 200, "chunk_overlap": 20},
    {"chunk_size": 200, "chunk_overlap": 50},
    {"chunk_size": 300, "chunk_overlap": 50}
]

for config in recursive_configs:
    chunk_size = config["chunk_size"]
    chunk_overlap = config["chunk_overlap"]
    print(f"\n--- Recursive Chunking (Chunk Size = {chunk_size}, Chunk Overlap = {chunk_overlap}) ---")
    chunks = recursive_chunking(original_document_text, chunk_size, chunk_overlap)
    for i, doc_chunk in enumerate(chunks):
        print(f"Chunk Number: {i+1}")
        print(f"Chunk Content: {doc_chunk.page_content.strip()}")
        print("\n")


--- Recursive Chunking (Chunk Size = 200, Chunk Overlap = 20) ---
Chunk Number: 1
Chunk Content: Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy


Chunk Number: 2
Chunk Content: Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.


Chunk Number: 3
Chunk Content: Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.



--- Recursive Chunking (Chunk Size = 200, Chunk Overlap = 50) ---
Chunk Number: 1
Chunk Content: Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy


Chunk Number: 2
Chunk Content: Wo

#### Analysis of Recursive Character Chunking

**How overlap improves context retention:**

Recursive character chunking attempts to split text into semantically meaningful units by trying different separators (like newline, space, then character). When combined with chunk overlap, it significantly improves context retention in the following ways:

1.  **Reduces Context Loss at Boundaries:** By including a portion of the previous chunk in the current chunk, overlap ensures that sentences or ideas that span across chunk boundaries are not entirely cut off. This means that if a query relates to information at the end of one chunk and the beginning of the next, both parts of the context are present in at least one chunk.
2.  **Provides Redundancy for Retrieval:** Overlapping chunks create a redundancy of information. If a retrieval system misses a specific chunk, it might still find the relevant information in an overlapping chunk. This increases the chances of retrieving complete and coherent information for a given query.
3.  **Maintains Flow and Cohesion:** For RAG systems, having overlapping context helps the Language Model (LLM) understand the flow of information. If a retrieved chunk starts with an incomplete thought, the overlap provides the preceding context, allowing the LLM to better interpret and utilize the information.

### Task 4: Sentence-Based Chunking

**Objective:** Implement Sentence Chunking.

In [8]:
# Install NLTK
!pip install -q nltk
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Download the missing resource

from nltk.tokenize import sent_tokenize

# Split document into sentences
sentences = sent_tokenize(original_document_text)

print("--- Sentence-Based Chunking --")
for i, sentence in enumerate(sentences):
    print(f"Sentence Number: {i+1}")
    print(f"Sentence Content: {sentence.strip()}")
    print("\n")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- Sentence-Based Chunking --
Sentence Number: 1
Sentence Content: Leave Policy
Employees receive 12 casual leaves annually.


Sentence Number: 2
Sentence Content: Employees receive 15 sick leaves annually.


Sentence Number: 3
Sentence Content: Unused casual leaves cannot be carried forward.


Sentence Number: 4
Sentence Content: Work From Home Policy
Employees may work from home twice per week.


Sentence Number: 5
Sentence Content: Manager approval is required for additional remote work.


Sentence Number: 6
Sentence Content: Travel Policy
Travel expenses are reimbursed within 30 days.


Sentence Number: 7
Sentence Content: Original receipts must be submitted for reimbursement.


Sentence Number: 8
Sentence Content: Medical Insurance Policy
All employees are covered under company medical insurance.


Sentence Number: 9
Sentence Content: Dependent coverage is available for spouse and children.




#### Analysis of Sentence-Based Chunking

**Does sentence chunking preserve meaning better than fixed-size chunking?**

Yes, sentence chunking generally preserves meaning better than fixed-size chunking for several key reasons:

1.  **Linguistic Integrity:** Each chunk is a complete sentence, which is a fundamental unit of thought and meaning in language. This ensures that the linguistic integrity of the text is maintained, unlike fixed-size chunks that can arbitrarily cut off sentences mid-way.
2.  **Contextual Completeness:** A sentence typically conveys a single, complete idea or piece of information. By chunking at the sentence level, the system ensures that each retrieved chunk is a coherent and contextually complete statement, making it easier for a retrieval system or a language model to understand.
3.  **Improved Readability and Interpretability:** Chunks composed of full sentences are much more readable and interpretable for both humans and AI models. This clarity can lead to more accurate retrieval and better summarization or response generation in RAG systems.
4.  **Reduced Ambiguity:** When sentences are split, the partial information can become ambiguous. Sentence chunking avoids this by keeping related words and clauses together, reducing the chance of misinterpretation.

### Task 5: Semantic Chunking

**Objective:** Group content based on meaning.

In [9]:
import re

def semantic_chunking(text):
    chunks = []
    # Define patterns for policy sections
    # This simple approach assumes section titles are followed by their content
    patterns = {
        "Leave Policy": r"(Leave Policy.*?)(?=(Work From Home Policy|Travel Policy|Medical Insurance Policy|$))",
        "Work From Home Policy": r"(Work From Home Policy.*?)(?=(Leave Policy|Travel Policy|Medical Insurance Policy|$))",
        "Travel Policy": r"(Travel Policy.*?)(?=(Leave Policy|Work From Home Policy|Medical Insurance Policy|$))",
        "Medical Insurance Policy": r"(Medical Insurance Policy.*?)(?=(Leave Policy|Work From Home Policy|Travel Policy|$))",
    }

    processed_text = text # Keep track of what's been chunked

    # Iterate through patterns to find and extract chunks
    for title, pattern in patterns.items():
        match = re.search(pattern, processed_text, re.DOTALL)
        if match:
            chunk_content = match.group(1).strip()
            chunks.append({"title": title, "content": chunk_content})
            # Remove the found content to prevent re-matching in other patterns
            processed_text = processed_text.replace(chunk_content, "").strip()

    # A more robust approach could involve using NLTK for sentence tokenization and then embedding
    # sentences to group them semantically, but for this specific document structure,
    # regex-based section extraction works well to achieve the 'semantic' grouping by policy.

    return chunks

# Perform semantic chunking
semantic_chunks = semantic_chunking(original_document_text)

print("--- Semantic Chunking ---")
for i, chunk in enumerate(semantic_chunks):
    print(f"Semantic Chunk Number: {i+1}")
    print(f"Semantic Chunk Content:\n{chunk['content']}\n")

--- Semantic Chunking ---
Semantic Chunk Number: 1
Semantic Chunk Content:
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.

Semantic Chunk Number: 2
Semantic Chunk Content:
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.

Semantic Chunk Number: 3
Semantic Chunk Content:
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.

Semantic Chunk Number: 4
Semantic Chunk Content:
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.



#### Analysis of Semantic Chunking

**Why semantic chunking may improve retrieval quality:**

Semantic chunking aims to group text based on its meaning or thematic coherence, rather than arbitrary size limits or linguistic structures. For RAG systems, this approach can significantly improve retrieval quality for the following reasons:

1.  **Contextual Completeness for Topics:** Each semantic chunk is designed to contain a complete thought, concept, or section related to a specific topic (e.g., 'Leave Policy'). When a user's query relates to that topic, retrieving the entire semantic chunk ensures that all relevant information on that subject is provided, leading to more comprehensive and accurate answers.
2.  **Reduced Irrelevance:** By keeping unrelated information separate, semantic chunking minimizes the chance of retrieving chunks that contain only partially relevant information mixed with irrelevant text. This reduces noise and helps the language model focus on the core topic of the query.
3.  **Better Alignment with User Intent:** Users typically query for information about specific topics or concepts. Semantic chunks, by their nature, align well with these informational units, making it more likely that the retrieved chunk directly addresses the user's intent.
4.  **Improved Embedding Quality:** When generating embeddings for chunks, semantic chunks—being coherent and topic-focused—will likely have more distinct and accurate representations in the embedding space. This can lead to more precise similarity matches during retrieval.
5.  **Enhanced RAG Performance:** A language model receiving a semantically coherent chunk will have an easier time synthesizing a relevant and accurate answer, as it doesn't have to piece together information from fragmented chunks or filter out irrelevant content within a chunk.

### Task 6: Chunk Comparison

**Objective:** Compare all chunking techniques.

| Chunking Method      | Context Preservation                                       | Retrieval Quality                                                     | Implementation Complexity                                     |
| :------------------- | :--------------------------------------------------------- | :-------------------------------------------------------------------- | :------------------------------------------------------------ |
| **Fixed Size**       | Low (arbitrary splits can break context)                   | Low (can retrieve fragmented information)                             | Low (simple slicing)                                          |
| **Recursive**        | Medium (better than fixed, especially with overlap)        | Medium (overlap helps maintain context, but still can split ideas)  | Medium (requires understanding of separators and chunking logic) |
| **Sentence**         | High (each chunk is a complete thought)                    | High (retrieves coherent units of information)                        | Medium (requires NLP libraries like NLTK/SpaCy)               |
| **Semantic**         | Very High (groups content by meaning/topic)                | Very High (retrieves complete thematic sections, highly relevant)     | High (requires defining semantic boundaries or using advanced NLP/embeddings) |


### Task 7: Retrieval Simulation

**Objective:** Test retrieval effectiveness.

In [10]:
import re

# Define the queries
queries = [
    "How many casual leaves are provided?",
    "Can employees work from home?",
    "What is the travel reimbursement process?",
    "Who is covered under medical insurance?"
]

def simple_keyword_retrieval(query, chunks):
    best_chunk = "No relevant chunk found."
    max_score = 0
    query_keywords = set(re.findall(r'\b\w+\b', query.lower()))

    for chunk_obj in chunks:
        # Handle different chunk formats (LangChain Document vs. simple string vs. dict)
        if isinstance(chunk_obj, str):
            chunk_content = chunk_obj
        elif hasattr(chunk_obj, 'page_content'): # LangChain Document
            chunk_content = chunk_obj.page_content
        elif isinstance(chunk_obj, dict) and 'content' in chunk_obj: # Semantic chunk
            chunk_content = chunk_obj['content']
        else:
            chunk_content = str(chunk_obj) # Fallback

        chunk_lower = chunk_content.lower()
        score = sum(1 for keyword in query_keywords if keyword in chunk_lower)

        if score > max_score:
            max_score = score
            best_chunk = chunk_content

    return best_chunk.strip()

# Prepare chunks for retrieval simulation

# Fixed Size Chunks (using chunk_size=200 as representative)
fixed_size_chunks_200 = fixed_size_chunking(original_document_text, 200)

# Recursive Chunks (using chunk_size=200, chunk_overlap=50 as representative)
recursive_chunks_200_50 = recursive_chunking(original_document_text, 200, 50)

# Sentence Chunks (already available from previous task)
# sentences list is already flat, so no need to extract .page_content

# Semantic Chunks (already available from previous task)
# semantic_chunks are dicts, retrieval function handles this

chunking_strategies = {
    "Fixed Size (size=200)": fixed_size_chunks_200,
    "Recursive (size=200, overlap=50)": recursive_chunks_200_50,
    "Sentence-Based": sentences,
    "Semantic": semantic_chunks
}

print("--- Retrieval Simulation ---\n")

for query in queries:
    print(f"Query: {query}\n")
    for method_name, chunks_list in chunking_strategies.items():
        retrieved_chunk = simple_keyword_retrieval(query, chunks_list)
        print(f"  Chunking Method: {method_name}")
        print(f"  Retrieved Chunk: {retrieved_chunk}\n")
    print("----------------------------------------\n")


--- Retrieval Simulation ---

Query: How many casual leaves are provided?

  Chunking Method: Fixed Size (size=200)
  Retrieved Chunk: Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home

  Chunking Method: Recursive (size=200, overlap=50)
  Retrieved Chunk: Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy

  Chunking Method: Sentence-Based
  Retrieved Chunk: Leave Policy
Employees receive 12 casual leaves annually.

  Chunking Method: Semantic
  Retrieved Chunk: Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.

----------------------------------------

Query: Can employees work from home?

  Chunking Method: Fixed Size (s

### Task 8: Recommendation Report

**Objective:** Act as an AI Engineer and prepare a report answering: Which chunking strategy would you use for: HR Policy Assistant? Legal Document Assistant? Medical Document Assistant? Research Paper Assistant? Justify your answers.

#### Recommendation Report on Document Chunking Strategies

As an AI Engineer, the choice of chunking strategy for a Retrieval-Augmented Generation (RAG) system is critical for optimizing retrieval quality and ultimately the effectiveness of the AI assistant. Based on our analysis and retrieval simulation, here are recommendations for different types of document assistants:

---

1.  **HR Policy Assistant**
    *   **Recommended Strategy: Semantic Chunking**
    *   **Justification:** HR policy documents (like the one we used) are typically structured around distinct topics (e.g., Leave Policy, Work From Home Policy, Medical Insurance Policy). Semantic chunking, especially when guided by section headers or natural breaks in policy, ensures that each chunk represents a complete and coherent policy statement. This leads to highly accurate retrieval of entire policy sections relevant to a user's query, as demonstrated in our simulation. Fixed-size or recursive chunking would risk splitting crucial policy details, and even sentence-based might retrieve too granular information without the full context of a policy. For an HR assistant, providing a complete policy context is paramount.

---

2.  **Legal Document Assistant**
    *   **Recommended Strategy: Semantic Chunking (with sub-section awareness) or Sentence-Based Chunking (for highly granular queries)**
    *   **Justification:** Legal documents often have a hierarchical structure (articles, sections, sub-sections) and require extreme precision. **Semantic chunking** that respects these inherent legal structures (e.g., using regex to identify specific articles or clauses) would be ideal to ensure complete legal arguments or definitions are retrieved. However, legal queries can also be highly granular, sometimes needing a specific sentence that defines a term or condition. In such cases, **Sentence-Based Chunking** offers excellent precision by ensuring each retrieved unit is a linguistically complete statement, minimizing the risk of misinterpretation. A hybrid approach, or a system that can dynamically select between these, might be most effective.

---

3.  **Medical Document Assistant**
    *   **Recommended Strategy: Semantic Chunking (based on clinical concepts/sections) or Recursive Chunking (with high overlap for complex narratives)**
    *   **Justification:** Medical documents (e.g., patient records, research papers) are often complex, containing detailed narratives, observations, and findings. **Semantic chunking** that can identify and group information by clinical concepts (e.g., 'Patient History', 'Diagnosis', 'Treatment Plan', 'Medication') would be highly beneficial for retrieving complete medical contexts. However, if the content is more free-flowing narrative (like doctor's notes or research descriptions), **Recursive Chunking with a high overlap** would be valuable. The overlap helps maintain the continuity of complex medical descriptions and ensures that no critical diagnostic details or symptoms are split across chunks, which could lead to missed information or misdiagnosis.

---

4.  **Research Paper Assistant**
    *   **Recommended Strategy: Recursive Chunking (with moderate to high overlap) or Semantic Chunking (based on paper sections)**
    *   **Justification:** Research papers are structured with sections like 'Abstract', 'Introduction', 'Methodology', 'Results', 'Discussion', 'Conclusion'. **Semantic chunking** aligning with these sections would allow for retrieval of complete arguments or descriptions of a particular aspect of the research. However, within these sections, the content is dense and highly interconnected. **Recursive Chunking** is excellent here because it handles longer, more complex texts by trying to split on various delimiters and can maintain context through overlap. A moderate to high overlap would be crucial to ensure that detailed scientific explanations or experimental setups are not fragmented, which could break the logical flow required to understand complex scientific concepts.

---

In conclusion, while Fixed Size Chunking is easy to implement, its arbitrary nature makes it generally unsuitable for robust RAG systems where context and meaning are paramount. Sentence-Based Chunking offers good linguistic integrity but might be too granular for topics requiring broader context. Recursive and especially Semantic Chunking prove to be more effective by aligning chunks with the inherent structure and meaning of the document, thereby significantly improving the quality of information retrieval for AI assistants.